## 02 - Data Preprocessing

In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

In [2]:
df = pd.read_csv('Data/combined_sold.csv')

/var/folders/jn/2yd4xw3j43l6tt2w0gfb6nh80000gn/T/ipykernel_71196/69749230.py:1: DtypeWarning: Columns (0,1,4,9,78,79,80,81,82) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('Data/combined_sold.csv')


In [3]:
df.isnull().sum()

BuyerAgentAOR                   37349
ListAgentAOR                    33976
Flooring                       107944
ViewYN                          27063
WaterfrontYN                   302202
                                ...  
lonfilled                      254434
OriginatingSystemName          260912
OriginatingSystemSubName       260912
BuyerAgencyCompensationType    268392
BuyerAgencyCompensation        268404
Length: 84, dtype: int64

In [4]:
# removing columns that have over 80% missing/null values

def get_nan_cols(df, thresh=0.8):
    threshold = len(df.index) * thresh
    return [c for c in df.columns if df[c].isnull().sum() >= threshold]

print(f"Columns With Over 80% Missing Values: {get_nan_cols(df)}")

df = df.dropna(axis=1, thresh=0.2*len(df))

Columns With Over 80% Missing Values: ['WaterfrontYN', 'BasementYN', 'FireplacesTotal', 'AboveGradeFinishedArea', 'TaxAnnualAmount', 'ElementarySchool', 'BuilderName', 'TaxYear', 'BuildingAreaTotal', 'ElementarySchoolDistrict', 'CoBuyerAgentFirstName', 'BelowGradeFinishedArea', 'BusinessType', 'CoveredSpaces', 'MiddleOrJuniorSchool', 'HighSchool', 'LotSizeDimensions', 'MiddleOrJuniorSchoolDistrict', 'latfilled', 'lonfilled', 'OriginatingSystemName', 'OriginatingSystemSubName', 'BuyerAgencyCompensationType', 'BuyerAgencyCompensation']


In [5]:
# drop these 2 columns since they have no more predictive power after filitering to single family homes
df = df.drop(columns=['PropertyType', 'PropertySubType'])


# drop columns that are solely used for identification and/or have no predictive power
df = df.drop(columns = ['ListAgentFullName', 'ListAgentFirstName', 'ListAgentLastName','ListAgentEmail', 
                        'CoListAgentFirstName', 'CoListAgentLastName', 'BuyerAgentFirstName', 'BuyerAgentLastName', 'BuyerAgentMlsId',
                        'BuyerAgentAOR', 'ListAgentAOR', 'BuyerOfficeAOR', 'ListOfficeName', 'BuyerOfficeName', 'CoListOfficeName',
                        'ListingKey', 'ListingId', 'ListingKeyNumeric', 'AssociationFeeFrequency'])

df = df.drop(columns = [])


In [6]:
df.columns

Index(['Flooring', 'ViewYN', 'PoolPrivateYN', 'OriginalListPrice', 'CloseDate',
       'ClosePrice', 'Latitude', 'Longitude', 'UnparsedAddress', 'LivingArea',
       'ListPrice', 'DaysOnMarket', 'MLSAreaMajor', 'CountyOrParish',
       'MlsStatus', 'AttachedGarageYN', 'ParkingTotal', 'LotSizeAcres',
       'SubdivisionName', 'YearBuilt', 'StreetNumberNumeric',
       'BathroomsTotalInteger', 'City', 'BedroomsTotal',
       'ContractStatusChangeDate', 'PurchaseContractDate',
       'ListingContractDate', 'StateOrProvince', 'FireplaceYN', 'Stories',
       'Levels', 'LotSizeArea', 'MainLevelBedrooms', 'NewConstructionYN',
       'GarageSpaces', 'HighSchoolDistrict', 'PostalCode', 'AssociationFee',
       'LotSizeSquareFeet'],
      dtype='object')

In [7]:
df.isnull().sum()

Flooring                    107944
ViewYN                       27063
PoolPrivateYN                29314
OriginalListPrice              593
CloseDate                        0
ClosePrice                       2
Latitude                     11174
Longitude                    11174
UnparsedAddress                287
LivingArea                     157
ListPrice                        0
DaysOnMarket                     0
MLSAreaMajor                 43099
CountyOrParish                   0
MlsStatus                        0
AttachedGarageYN             35107
ParkingTotal                   344
LotSizeAcres                  5214
SubdivisionName             197527
YearBuilt                      218
StreetNumberNumeric            355
BathroomsTotalInteger           50
City                           243
BedroomsTotal                    0
ContractStatusChangeDate       452
PurchaseContractDate           137
ListingContractDate              0
StateOrProvince                  0
FireplaceYN         

In [8]:
# since theres only 2 rows with missing ClosePrice and 1 row with missing PostalCode, drop those rows
df = df.dropna(subset=['ClosePrice', 'PostalCode'])

#filling in some null values
cols = ['LivingArea', 'BedroomsTotal', 'BathroomsTotalInteger', 'LotSizeArea','LotSizeAcres', 'LotSizeSquareFeet', 'MainLevelBedrooms', 'ParkingTotal']
for col in cols:
    median = df[col].median()
    df[col] = df[col].fillna(median)

cols = ['Flooring', 'UnparsedAddress', 'MLSAreaMajor', 'SubdivisionName', 'HighSchoolDistrict', 'Levels', 'City']
for col in cols:
    df[col] = df[col].fillna('Unknown')


df['AssociationFee'] = df['AssociationFee'].fillna(0)
df['GarageSpaces'] = df['GarageSpaces'].fillna(0)

# assume nulls means 1 stories
df['Stories'] = df['Stories'].fillna(1.0)

In [9]:
# converting categorical fields

binary_cols = ['PoolPrivateYN', 'ViewYN', 'FireplaceYN', 'NewConstructionYN', 'AttachedGarageYN']

for col in binary_cols:
    df[col] = df[col].map({'Y': 1, 'N': 0, True: 1, False: 0})
    
    # fill remaining null with 0 
    df[col] = df[col].fillna(0).astype(int)


In [10]:
df['Levels'].value_counts()

Levels
One                               169563
Two                                92157
Unknown                            28776
ThreeOrMore                         5701
MultiSplit                          4203
One,Two                              641
Two,MultiSplit                       445
Two,One                              264
ThreeOrMore,MultiSplit               204
One,MultiSplit                       167
Two,ThreeOrMore                      103
One,ThreeOrMore                       46
MultiSplit,One                        26
One,Two,MultiSplit                     9
Two,ThreeOrMore,MultiSplit             7
One,Two,ThreeOrMore                    6
ThreeOrMore,One                        6
One,Two,ThreeOrMore,MultiSplit         3
Two,MultiSplit,One                     2
Name: count, dtype: int64

In [11]:
def clean_to_single_level(level_str):
    if pd.isna(level_str):
        return 'Unknown'
    
    level_str = str(level_str)
    
    # Prioritize the most significant structural trait
    if 'ThreeOrMore' in level_str:
        return 'ThreeOrMore'
    elif 'Two' in level_str:
        return 'Two'
    elif 'MultiSplit' in level_str:
        return 'MultiSplit'
    elif 'One' in level_str:
        return 'One'
    else:
        return 'Other'

# Apply the cleaning function
df['Levels'] = df['Levels'].apply(clean_to_single_level)

df = pd.get_dummies(df, columns=['Levels'], drop_first=True, dtype=int)

In [12]:
df.isnull().sum().sort_values(ascending=False)

Latitude                    11174
Longitude                   11174
OriginalListPrice             593
ContractStatusChangeDate      452
StreetNumberNumeric           355
YearBuilt                     218
PurchaseContractDate          137
MainLevelBedrooms               0
ListingContractDate             0
StateOrProvince                 0
FireplaceYN                     0
Stories                         0
LotSizeArea                     0
Flooring                        0
NewConstructionYN               0
GarageSpaces                    0
PostalCode                      0
AssociationFee                  0
LotSizeSquareFeet               0
Levels_One                      0
Levels_Other                    0
Levels_ThreeOrMore              0
HighSchoolDistrict              0
BathroomsTotalInteger           0
BedroomsTotal                   0
City                            0
PoolPrivateYN                   0
CloseDate                       0
ClosePrice                      0
UnparsedAddres

In [13]:
# normalizing key numerical fields

num_fields = ['LivingArea', 'LotSizeArea', 'LotSizeSquareFeet', 'LotSizeAcres']
df.loc[:, num_fields] = np.log1p(df[num_fields])

In [14]:
df.head()

,Flooring,ViewYN,PoolPrivateYN,OriginalListPrice,CloseDate,ClosePrice,Latitude,Longitude,UnparsedAddress,LivingArea,...,NewConstructionYN,GarageSpaces,HighSchoolDistrict,PostalCode,AssociationFee,LotSizeSquareFeet,Levels_One,Levels_Other,Levels_ThreeOrMore,Levels_Two
0,Unknown,0,0,1130000.0,2024-09-12,1090000.0,34.136660,-118.012799,626 Montana Street,7.884577,...,0,0.0,Monrovia Unified,91016,0.0,9.574358,1,0,0,0
1,Tile,1,1,1995000.0,2024-09-30,1995000.0,33.804370,-116.439882,334 Loch Lomond Road,8.167636,...,0,3.0,Unknown,92270,850.0,9.814438,1,0,0,0
2,Wood,1,1,2340000.0,2024-09-30,2340000.0,33.515988,-117.706900,30812 Palmetto Place,8.002694,...,0,2.0,Capistrano Unified,92677,240.0,9.384378,0,0,0,1
3,Unknown,1,1,984000.0,2024-09-30,984000.0,32.799141,-117.017206,8337 Lake Baca,7.361375,...,0,2.0,San Diego Unified,92119,0.0,8.779711,1,0,0,0
4,"Carpet,Stone,Tile",1,0,1250000.0,2024-09-30,1225000.0,34.024425,-117.136006,Unknown,7.969012,...,0,2.5,Redlands Unified,92373,425.0,8.582794,1,0,0,0


In [15]:
df.to_csv('Data/cleaned_sold.csv', index=False)

### Create Train/Test Split

In [ ]:
#testing reusable function to see if it works

from utils import create_time_split

df = pd.read_csv('Data/cleaned_sold.csv')
df['CloseDate'] = pd.to_datetime(df['CloseDate'])

max_date = df['CloseDate'].max()
test_start_date = max_date - pd.DateOffset(months=1)

train_df, test_df = create_time_split(df, 'CloseDate', X_months=6, test_start_date=test_start_date, max_date=max_date)

Training Window (X=6 months): 2025-09-30 to 2026-03-30 | Rows: 52109
Testing Window (1 month): 2026-03-30 to 2026-04-30
